# HandE_Tests — Robotiq Hand-E (PolyScope X) gripper bring-up

Gripper-only sanity checks for the Hand-E on the lab **UR3e / PolyScope X** (`192.168.1.4`),
using `HandEGripper` from `HandE_dependency.py` (this folder) — the single source of gripper
truth that `UR3RealRobotPick` also delegates to.

PolyScope X exposes the gripper via a **Robotiq URCapX XML-RPC server** on
`http://<host>:49999/` (NOT the legacy 63352 socket); this Hand-E is **slaveId 9**. This
channel is independent of the arm — see `robots/UR3e/UR3_RTDE_Tests.ipynb`.

1. Connect + read native state (no motion)
2. Open / close test — locks the sim↔percent direction
3. Sim-value command check

Sim convention (per finger): **`0` = OPEN, `0.025` = CLOSED**; native percent
**`0 %` = OPEN, `100 %` = CLOSED**. ⚠️ Don't command the XML-RPC server above ~10 Hz.

In [ ]:
# Setup: self-locate HandE_dependency.py, build the gripper handle
import os
import sys
import time

# regardless of kernel cwd (VSCode runs notebooks from the workspace root).
for _d in (os.getcwd(), os.path.join(os.getcwd(), "robots", "hande"),
           os.path.abspath(os.path.join(os.getcwd(), "..", "hande"))):
    if os.path.exists(os.path.join(_d, "HandE_dependency.py")):
        sys.path.insert(0, _d)
        break
else:
    raise FileNotFoundError("HandE_dependency.py not found on any candidate path")
from HandE_dependency import HandEGripper

ROBOT_IP = "192.168.1.4"   # UR3e / PolyScope X
SLAVE_ID = 9               # this Hand-E = slaveId 9 ("Gripper ID 1" in the UI)

g = HandEGripper(ROBOT_IP, slave_id=SLAVE_ID)
print("HandEGripper handle ready for", ROBOT_IP, "slaveId", SLAVE_ID)

## 1. Connect + read native state

`connect()` opens the XML-RPC server and activates the gripper (`activateIfRequired`).

> If it raises `Fault 10: ... Gripper with ID 9 not found`, the Hand-E isn't scanned under
> that slaveId — open the Robotiq URCapX on the pendant, confirm the gripper is detected, and
> set `SLAVE_ID` to match.

In [ ]:
# 1 — connect + activate, then read native state (no motion)
g.connect()   # XML-RPC :49999, activateIfRequired

state = g.read_state()
print("native (URCapX, position in percent 0-100):")
for k in ["pos_pct", "obj_flag", "grasped", "fault", "activated", "connected"]:
    print(f"  {k:10s} = {state[k]}")
print(f"  sim_finger = {state['sim_finger']:.4f}  (per-finger m; 0=open, 0.025=closed)")

## 2. Open / close test — lock the direction

Drives the direction-independent `open_gripper()` / `close_gripper()` and records the native
`pos_pct` at each extreme. This **locks the percent direction**: if OPEN reports ~100 % and
CLOSED ~0 %, flip `HandEGripper.NATIVE_OPEN_PCT` / `NATIVE_CLOSED_PCT` in `HandE_dependency.py`
(the only constants that encode direction).

In [ ]:
# 2 — open / close, recording native pos_pct at each extreme
print("opening...")
g.open_gripper()
time.sleep(2.0)
open_pct = g.read_state()["pos_pct"]
print("  pos_pct when OPEN  :", open_pct)

print("closing...")
g.close_gripper()
time.sleep(2.0)
closed_pct = g.read_state()["pos_pct"]
print("  pos_pct when CLOSED:", closed_pct)

print("\nDirection check (defaults NATIVE_OPEN_PCT=0, NATIVE_CLOSED_PCT=100):")
print(f"  pos_pct @ OPEN   = {open_pct}   (expect ~0)")
print(f"  pos_pct @ CLOSED = {closed_pct}   (expect ~100)")
print("  -> if reversed, flip those two class constants in HandE_dependency.py")

## 3. Sim-value command check

`command(sim_value)` takes a per-finger sim position in metres `[0, 0.025]` (0 = open,
0.025 = closed) and maps it to native percent via the verified mapping. Step through a few
values and read back the position.

In [ ]:
# 3 — sim-value sweep (0 open -> 0.025 closed)
for sim in (0.0, 0.0125, 0.025):
    pct = g.command(sim)
    time.sleep(1.0)
    st = g.read_state()
    print(f"  command(sim={sim:.4f}) -> {pct:5.1f}% | readback pos_pct={st['pos_pct']:.1f} "
          f"sim_finger={st['sim_finger']:.4f} grasped={st['grasped']}")

## Teardown

Drops the XML-RPC handle (leaves the gripper activated on the controller).

In [ ]:
g.disconnect()
print("gripper disconnected")